In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from pathlib import Path
import sys
import os

ROOT_DIR = Path().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from scripts.helpers.datasets import load_taxi_data

In [2]:
df = load_taxi_data(preprocessed=False)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6593828 entries, 0 to 6593827
Data columns (total 21 columns):
 #   Column                      Dtype                          
---  ------                      -----                          
 0   trip_id                     str                            
 1   taxi_id                     str                            
 2   trip_start_timestamp        datetime64[us, America/Chicago]
 3   trip_end_timestamp          datetime64[us, America/Chicago]
 4   trip_seconds                int64                          
 5   trip_miles                  float64                        
 6   pickup_census_tract         int64                          
 7   dropoff_census_tract        int64                          
 8   pickup_community_area       float64                        
 9   dropoff_community_area      float64                        
 10  fare                        float64                        
 11  tips                        float64             

In [5]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude
0,22208de57c456e7e6ea5f60bdc1746ad300535a9,04b96cbbdcfe5b7cbb6884bc1b922819466f652662ead8...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,2214,18.23,17031980000,17031320100,76.0,32.0,...,7.91,0.0,4.0,60.66,Credit Card,5 Star Taxi,41.979071,-87.903040,41.884987,-87.620993
1,35968e44a8ea32a0849720b91c35a4d5a8ff6484,4a991432c3e0600b9c919a01148b17b866d29a41751b95...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,120,0.00,17031081600,17031081201,8.0,8.0,...,0.00,0.0,0.0,3.75,Cash,Taxi Affiliation Services,41.892073,-87.628874,41.899156,-87.626211
2,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,2024-01-01 00:00:00-06:00,2024-01-01 00:30:00-06:00,1681,15.34,17031980000,17031071400,76.0,7.0,...,8.85,0.0,4.0,53.10,Credit Card,Globe Taxi,41.979071,-87.903040,41.922083,-87.634156
3,541cf2b862280d13b36e466ad90d9485e1ae1600,13c8f729e7e5a9f850e406e3b31524c6881649044dab76...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,59,0.00,17031980000,17031980000,76.0,76.0,...,0.00,0.0,0.0,3.50,Cash,5 Star Taxi,41.979071,-87.903040,41.979071,-87.903040
4,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,2024-01-01 00:00:00-06:00,2024-01-01 00:00:00-06:00,180,0.30,17031081500,17031081201,8.0,8.0,...,0.00,0.0,1.0,5.25,Cash,"Taxicab Insurance Agency, LLC",41.892508,-87.626215,41.899156,-87.626211


In [6]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'dropoff_centroid_latitude',
       'dropoff_centroid_longitude'],
      dtype='str')

## Handle missing values




In [7]:
print(df.isnull().sum())

trip_id                           0
taxi_id                           4
trip_start_timestamp            104
trip_end_timestamp              111
trip_seconds                      0
trip_miles                        0
pickup_census_tract               0
dropoff_census_tract              0
pickup_community_area          7538
dropoff_community_area        80242
fare                              0
tips                              0
tolls                             0
extras                            0
trip_total                        0
payment_type                      0
company                           0
pickup_centroid_latitude          0
pickup_centroid_longitude         0
dropoff_centroid_latitude         0
dropoff_centroid_longitude        0
dtype: int64


Most of the missing values are the community area. As the census tract corresponds with the community we can use the mapping from census tract and community area, which is available online to fill in the missing values. Also some timestamps are missing.

### Missing Timestamps
First we try to identify if the missing timestamps came from parsing them to the local time in Chicago.

In [8]:
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
raw_ts = pd.read_csv(RAW_DATA_DIR / "chicago_taxi_trips_2024.csv", 
                      usecols=["trip_start_timestamp"])
nat_rows = df[df["trip_start_timestamp"].isna()].index
print(raw_ts.loc[nat_rows, "trip_start_timestamp"].unique())

<StringArray>
['2024-11-03T01:00:00.000', '2024-11-03T01:15:00.000',
 '2024-11-03T01:30:00.000', '2024-11-03T01:45:00.000',
 '2025-11-02T01:00:00.000', '2025-11-02T01:15:00.000',
 '2025-11-02T01:30:00.000', '2025-11-02T01:45:00.000']
Length: 8, dtype: str


The missing timestamps coincide with the daylight saving time transition from summer to winter time. Rather than being truly absent, these entries were simply dropped during parsing because they were ambiguous or invalid in that context. AS this is the case for a very small portion of trips, we can just drop these observations.

### Missing Community Areas
Chicago census tracts are fully nested within community areas, so every trip coordinate maps to exactly one community area. We use a spatial join against the official Chicago community area boundaries (downloaded via `scripts/download_community_areas.py`) to fill the missing values. Trips whose coordinates fall outside Chicago's 77 community areas (e.g. suburban drop-offs) will remain NaN and are kept as-is.

In [11]:
ca_gdf = gpd.read_file(ROOT_DIR / "data" / "raw" / "community_areas.geojson")[["area_numbe", "geometry"]]
ca_gdf.rename(columns={"area_numbe": "area_number"}, inplace=True)
ca_gdf["area_number"] = pd.to_numeric(ca_gdf["area_number"])
ca_gdf.head()

,area_number,geometry
0,1,"MULTIPOLYGON (((-87.65456 41.99817, -87.65574 ..."
1,2,"MULTIPOLYGON (((-87.68465 42.01948, -87.68464 ..."
2,3,"MULTIPOLYGON (((-87.64102 41.9548, -87.644 41...."
3,4,"MULTIPOLYGON (((-87.67441 41.9761, -87.6744 41..."
4,5,"MULTIPOLYGON (((-87.67336 41.93234, -87.67342 ..."


In [12]:
def fill_community_area(df, lat_col, lon_col, ca_col, ca_gdf):
    null_mask = df[ca_col].isna()
    if not null_mask.any():
        return df

    pts = gpd.GeoDataFrame(
        index=df.index[null_mask],
        geometry=gpd.points_from_xy(
            df.loc[null_mask, lon_col],
            df.loc[null_mask, lat_col],
        ),
        crs="EPSG:4326",
    )
    joined = pts.sjoin(ca_gdf, how="left", predicate="within")
    df.loc[null_mask, ca_col] = joined["area_number"].values
    return df

df = fill_community_area(df, "pickup_centroid_latitude",  "pickup_centroid_longitude",  "pickup_community_area",  ca_gdf)
df = fill_community_area(df, "dropoff_centroid_latitude", "dropoff_centroid_longitude", "dropoff_community_area", ca_gdf)

print(df[["pickup_community_area", "dropoff_community_area"]].isnull().sum())

pickup_community_area     0
dropoff_community_area    0
dtype: int64


### Missing taxiID

In [13]:
print(df.isnull().sum())

trip_id                         0
taxi_id                         4
trip_start_timestamp          104
trip_end_timestamp            111
trip_seconds                    0
trip_miles                      0
pickup_census_tract             0
dropoff_census_tract            0
pickup_community_area           0
dropoff_community_area          0
fare                            0
tips                            0
tolls                           0
extras                          0
trip_total                      0
payment_type                    0
company                         0
pickup_centroid_latitude        0
pickup_centroid_longitude       0
dropoff_centroid_latitude       0
dropoff_centroid_longitude      0
dtype: int64


## Consistency Checks

### Trips that didn't happen
Are the any trips where the start and end is at the same time and the pickup and dropoff is at the same place? These trips can be savely deleted as they don't produce any value and don't count as a real trip for our demand.

In [14]:
mask = (
    (df['trip_seconds'] == 0) &
    (df['trip_end_timestamp'] == df['trip_start_timestamp']) &
    (df['pickup_centroid_latitude'] == df['dropoff_centroid_latitude']) &
    (df['pickup_centroid_longitude'] == df['dropoff_centroid_longitude'])
)
mask.sum()

np.int64(83221)

### Trips with 0 miles

We will check for trips with 0 miles but where the duration of the trip is longer than 0 seconds

In [15]:
df_zero_miles = df[(df['trip_miles'] == 0) & (df['trip_seconds'] > 0)]
print(df_zero_miles['trip_seconds'].describe())

count    496082.000000
mean        478.124457
std        1399.234288
min           1.000000
25%           6.000000
50%          46.000000
75%         511.000000
max       86396.000000
Name: trip_seconds, dtype: float64


The 496,082 trips with `trip_miles == 0` and `trip_seconds > 0` are dropped. The durationn distribution reveals that the vast majority are not real trips: the median duration is only 46 seconds and the 25tgh percentile is just 6 seconds, indicating these are overwhelmingly meter-on/meter-off cancellations or GPS failures where movement was never recorded. A real taxitrip must cover some distance; retaining these rows would distort trip distance and speed features and corrupt spatial demand analysis since pickup and dropoff centroids coincide for GPS failures.

In [54]:
df = df[df['trip_miles'] > 0]
print(df.shape)

(6012652, 21)


In [11]:
mask_2 = (
    df['trip_miles'] == 0
)
mask_2.sum()

np.int64(575888)

In [12]:
mask_2 = (
    (df['trip_miles'] <= 0) &
    (df['trip_seconds'] > 0)
)
mask_2.sum()


np.int64(496082)

### Check for trips where the end of the trip is before the start

In [13]:
mask_3 = (
    df['trip_end_timestamp'] < df['trip_start_timestamp']
)
mask_3.sum()

np.int64(0)

## Handle Duplicates

### Check for duplicated trips on tridId

In [14]:
dup_df = df[df['trip_id'].duplicated(keep=False)]
print(f"Number of duplicate trip_id entries:", dup_df.shape[0])

Number of duplicate trip_id entries: 0


## Loading & Preprocessing in Subsequent Notebooks

All steps documented above are encapsulated in two helper scripts so that later notebooks can load a clean dataset in a single line:

- **`scripts/helpers/datasets.py`** — entry point for loading any dataset. `load_taxi_data(preprocessed=True)` reads the raw CSV and optionally applies the full preprocessing pipeline.
- **`scripts/helpers/preprocessing.py`** — contains `preprocess_taxi_data()`, which imputes missing community areas, drops invalid rows, and returns a clean `DataFrame`.

Usage in any subsequent notebook:

```python
from scripts.helpers.datasets import load_taxi_data, load_merged_data

df = load_taxi_data(preprocessed=True)       # cleaned dataset
df_raw = load_taxi_data(preprocessed=False)  # raw data only
df_merged = load_merged_data() # preprocessed taxi + weather data
```